In [ ]:
import pandas as pd
books = pd.read_csv("books_with_categories.csv")


In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)
classifier("I love this!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'joy', 'score': 0.9771687984466553}]

In [ ]:
import numpy as np
emotional_labels =["anger","disgust","joy","fear","sadness","suprise","neutral"]
isbn=[]
emotion_scores = {label: [] for label in emotional_labels}

def calculation_emotional_scores(predictions):
  per_emotion_scores = {label: [] for label in emotional_labels}
  for item in predictions:
    label = item["label"]
    score = item["score"]
    if label in per_emotion_scores:
      per_emotion_scores[label].append(score)

  max_scores = {}
  for label in emotional_labels:
    if per_emotion_scores[label]:
      max_scores[label] = np.max(per_emotion_scores[label])
    else:
      max_scores[label] = 0.0
  return max_scores

In [ ]:
emotional_labels =["anger","disgust","joy","fear","sadness","suprise","neutral"]
isbn=[]
emotion_scores = {label: [] for label in emotional_labels}
from tqdm import tqdm

for i in tqdm(range(len(books))):
  isbn.append(books["isbn13"][i])
  sentences = books["description"][i].split(".")
  predictions = classifier(sentences)
  max_scores = calculation_emotional_scores(predictions)
  for label in emotional_labels:
    emotion_scores[label].append(max_scores[label])

 98%|█████████▊| 5072/5197 [28:16<00:37,  3.30it/s]

In [ ]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn


In [ ]:
books = pd.merge(books, emotions_df,on = "isbn13")


In [ ]:
books.to_csv("books_sentimental.csv",index = False)


In [ ]:
from google.colab import files
files.download()